In [ ]:
import json
import asyncio
from tqdm import tqdm
from ollama import AsyncClient
import os
import re
import nest_asyncio
nest_asyncio.apply()
client = AsyncClient()
src_paths = ['/home/qluai/zjs/TruthTorchLM-main/medmcqa/对比解码部分/my_method/ACD/qwen_data/qwen_seper/medmcqa_seper_results_with_retrieval.jsonl', '/home/qluai/zjs/TruthTorchLM-main/medmcqa/对比解码部分/my_method/ACD/qwen_data/qwen_seper/medqa/medqa_seper_results_with_retrieval.jsonl']
tgt_filenames = ['medmcqa_llama3.1_8b_cot_results.jsonl', 'medqa_llama3.1_8b_cot_results.jsonl']
model_name = 'llama3.1:8b-instruct-fp16'
output_dir = '/home/qluai/zjs/TruthTorchLM-main/llama3.1/8b_results/COT'
concurrency = 8
os.makedirs(output_dir, exist_ok=True)

def build_prompt(q):
    return f'Answer the following medical question with step-by-step reasoning.\n\nQuestion: {q}\n\nProvide your response in this format:\nReasoning: [brief step-by-step thinking]\nAnswer: [your final answer]'

def extract_final_answer(response):
    answer_match = re.search('Answer:\\s*(.+?)(?:\\n|$)', response, re.IGNORECASE)
    if answer_match:
        return answer_match.group(1).strip()
    final_match = re.search('Final Answer:\\s*(.+?)(?:\\n|$)', response, re.IGNORECASE)
    if final_match:
        return final_match.group(1).strip()
    sentences = response.strip().split('.')
    if sentences:
        return sentences[-1].strip()
    return response.strip()

def check_answer_correctness(model_answer, ground_truths):
    if not model_answer:
        return False
    model_ans_lower = model_answer.lower().strip()
    option_match = re.search('\\b([A-Da-d])\\b', model_ans_lower)
    if option_match:
        model_option = option_match.group(1).upper()
    else:
        model_option = None
    for gt in ground_truths:
        gt_lower = str(gt).lower().strip()
        if model_ans_lower == gt_lower:
            return True
        if gt_lower in model_ans_lower or model_ans_lower in gt_lower:
            return True
        if model_option and gt_lower == model_option.lower():
            return True
        if len(gt_lower) == 1 and gt_lower.upper() in ['A', 'B', 'C', 'D']:
            if gt_lower.upper() == model_option:
                return True
    return False

async def chat_with_llama_async(prompt, temperature=0, num_predict=256, top_p=1.0):
    try:
        response = await client.chat(model=model_name, messages=[{'role': 'user', 'content': prompt}], options={'temperature': temperature, 'num_predict': num_predict, 'top_p': top_p, 'stream': False})
        return response['message']['content'].strip()
    except Exception as e:
        print(f'API call error: {e}')
        return ''

async def process_batch(batch_items):
    tasks = [chat_with_llama_async(build_prompt(item['question'])) for item in batch_items]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    processed = []
    for (item, result) in zip(batch_items, results):
        if isinstance(result, Exception):
            full_response = ''
            final_answer = ''
        else:
            full_response = result
            final_answer = extract_final_answer(result)
        is_correct = check_answer_correctness(final_answer, item['ground_truths'])
        processed.append({'question': item['question'], 'model_answer_full': full_response, 'model_answer': final_answer, 'ground_truths': item['ground_truths'], 'is_correct': is_correct})
    return processed

async def process_dataset(src_path, tgt_path, total_lines):
    batch_items = []
    correct_count = 0
    total_count = 0
    pbar = tqdm(total=total_lines, desc=f'Processing {os.path.basename(src_path)} with CoT', unit='lines')
    with open(src_path, 'r', encoding='utf-8') as fin, open(tgt_path, 'w', encoding='utf-8') as fout:
        line_count = 0
        for line in fin:
            try:
                item = json.loads(line.strip())
                batch_items.append(item)
                line_count += 1
                if len(batch_items) == concurrency:
                    batch_results = await process_batch(batch_items)
                    for res in batch_results:
                        fout.write(json.dumps(res, ensure_ascii=False) + '\n')
                        total_count += 1
                        if res['is_correct']:
                            correct_count += 1
                    batch_items.clear()
                    current_accuracy = correct_count / total_count * 100 if total_count > 0 else 0
                    elapsed = pbar.format_dict.get('elapsed', 0)
                    speed = line_count / elapsed if elapsed > 0 else 0
                    eta_min = (total_lines - line_count) / speed / 60 if speed > 0 else float('inf')
                    pbar.set_postfix({'concurrency': concurrency, 'speed': f'{speed:.1f} l/s', 'accuracy': f'{current_accuracy:.1f}%', 'correct': f'{correct_count}/{total_count}', 'ETA': f'{eta_min:.1f} min' if eta_min != float('inf') else 'N/A'})
                    pbar.update(concurrency)
            except Exception as e:
                print(f'Skip line: {e}')
                line_count += 1
                pbar.update(1)
                continue
        if batch_items:
            batch_results = await process_batch(batch_items)
            for res in batch_results:
                fout.write(json.dumps(res, ensure_ascii=False) + '\n')
                total_count += 1
                if res['is_correct']:
                    correct_count += 1
            pbar.update(len(batch_results))
    pbar.close()
    return {'total': total_count, 'correct': correct_count, 'accuracy': correct_count / total_count * 100 if total_count > 0 else 0}

async def main():
    all_results = []
    for (i, src_path) in enumerate(src_paths):
        tgt_filename = tgt_filenames[i]
        tgt_path = os.path.join(output_dir, tgt_filename)
        with open(src_path, 'r', encoding='utf-8') as fin_count:
            total_lines = sum((1 for _ in fin_count))
        print(f"\n{'=' * 70}")
        print(f'Dataset {i + 1}/{len(src_paths)}: {os.path.basename(src_path)}')
        print(f'Total lines: {total_lines}')
        print(f"{'=' * 70}\n")
        stats = await process_dataset(src_path, tgt_path, total_lines)
        print(f"\n{'=' * 70}")
        print(f'✓ Completed: {os.path.basename(src_path)}')
        print(f'  Output: {tgt_path}')
        print(f"  Total questions: {stats['total']}")
        print(f"  Correct answers: {stats['correct']}")
        print(f"  Accuracy: {stats['accuracy']:.2f}%")
        print(f"{'=' * 70}\n")
        all_results.append({'dataset': os.path.basename(src_path), 'stats': stats})
    print(f"\n{'#' * 70}")
    print(f"{'OVERALL RESULTS':^70}")
    print(f"{'#' * 70}\n")
    total_all = sum((r['stats']['total'] for r in all_results))
    correct_all = sum((r['stats']['correct'] for r in all_results))
    accuracy_all = correct_all / total_all * 100 if total_all > 0 else 0
    for result in all_results:
        print(f"📊 {result['dataset']}:")
        print(f"   Accuracy: {result['stats']['accuracy']:.2f}% ({result['stats']['correct']}/{result['stats']['total']})")
    print(f"\n{'─' * 70}")
    print(f'🎯 Overall Accuracy: {accuracy_all:.2f}% ({correct_all}/{total_all})')
    print(f"{'#' * 70}\n")
    summary_path = os.path.join(output_dir, 'accuracy_summary.json')
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump({'model': model_name, 'datasets': all_results, 'overall': {'total': total_all, 'correct': correct_all, 'accuracy': accuracy_all}}, f, indent=2, ensure_ascii=False)
    print(f'📄 Summary saved to: {summary_path}')
await main()


Dataset 1/2: medmcqa_seper_results_with_retrieval.jsonl
Total lines: 5000



Processing medmcqa_seper_results_with_retrieval.jsonl with CoT: 100%|██████████| 5000/5000 [5:16:13<00:00,  3.79s/lines, concurrency=8, speed=0.3 l/s, accuracy=54.8%, correct=2741/5000, ETA=0.0 min]       



✓ Completed: medmcqa_seper_results_with_retrieval.jsonl
  Output: /home/qluai/zjs/TruthTorchLM-main/llama3.1/8b_results/COT/medmcqa_llama3.1_8b_cot_results.jsonl
  Total questions: 5000
  Correct answers: 2741
  Accuracy: 54.82%


Dataset 2/2: medqa_seper_results_with_retrieval.jsonl
Total lines: 5000



Processing medqa_seper_results_with_retrieval.jsonl with CoT: 100%|██████████| 5000/5000 [3:13:30<00:00,  2.32s/lines, concurrency=8, speed=0.4 l/s, accuracy=60.1%, correct=3003/5000, ETA=0.0 min]    


✓ Completed: medqa_seper_results_with_retrieval.jsonl
  Output: /home/qluai/zjs/TruthTorchLM-main/llama3.1/8b_results/COT/medqa_llama3.1_8b_cot_results.jsonl
  Total questions: 5000
  Correct answers: 3003
  Accuracy: 60.06%


######################################################################
                           OVERALL RESULTS                            
######################################################################

📊 medmcqa_seper_results_with_retrieval.jsonl:
   Accuracy: 54.82% (2741/5000)
📊 medqa_seper_results_with_retrieval.jsonl:
   Accuracy: 60.06% (3003/5000)

──────────────────────────────────────────────────────────────────────
🎯 Overall Accuracy: 57.44% (5744/10000)
######################################################################

📄 Summary saved to: /home/qluai/zjs/TruthTorchLM-main/llama3.1/8b_results/COT/accuracy_summary.json
